# Условная оптимизация

## Метод штрафных функций

В предыдущей работе нам было необходимо не только решить задачу оптимизации, но и учесть ораничения на возможные решения.

Как вы могли заметить, не все методы оптимизации из тех, которые доступны в пакете SciPy, поддерживают условную оптимизацию, при этом для некоторых задач одни методы более предпочтительны, чем другие, и может так оказаться, что для решения именно вашей задачи тот метод, который поддерживает условную оптимизацию, неприменим, а ограничения все-таки есть.

В этом случае на помощь приходят методы штрафных функций и барьерных поверхностей.

Основная идея у этих методов очень похожа, и заключается в том, что ограничения включаются прямо в оптимизируемую функцию, и делают ее тем более неоптимальной, чем дальше решение заходит в запрещенную область.

Разница между этими методами состоит в том, где находится исходная точка, с которой начинается поиск оптимального решения. В случае метода штрафных функций начальная точка не обязана находиться внутри зоны допустимых с точки зрения ограничений решений, а в случае барьерных функций - должна обязательно. Связано это с тем, что барьерные функции за пределами границы вновь уменьшаются, а штрафные функции проболжают возрастать тем больше, чем дальше за пределы границы вышла текущая точка.

Например, необходимо найти минимум функции $f(x) = (x - 2)^2 + 1$.

Мы знаем, что минимум данной функции находится в точке  $x = 2$, и значение минимизируемой функции в ней равно $1$.

Допустим, что существует ограничение $-\infty < x \le 1$. В этом случае минимум функции будет как раз в точке 1.

Попробуем ввести барьерную функцию. Требования к ней просты: в допустимой зоне ее значение должно быть близким к нулю, чтобы не оказывать влияния на целевую функцию, а в запрещенной зоне ее значение должно увеличиваться.

Введем функцию $g(x) = \frac {1} {1 - x}$. Ее особенность состоит в том, что внутри целевого диапазона она достаточно мала, а при приближении к границе (число 1), начинает возрастать (поскольку $\frac {1} / {1 - 1}$ становится равной бесконечности).

Таким образом, целевая функция теперь приобретает вот такой вид:

$$
f(x) = (x - 2)^2 + 1 + \frac {1} {1 - x}
$$

Казалось бы, все хорошо? На самом деле, нет.

Дело в том, что для данного случая проблема заключается вот в чем: минимум нашей фукнции находится в точке $x = 1$ (мы помним, что на смом деле в точке $x = 2$, но у нас есть ограничение $-\infty < x \le 1$, соответственно, минимум будет именно в точке $x = 1$). Однако, если мы используем ограничения...

In [ ]:
import numpy as np  # нам понадобится пакет numpy, чтобы использовать значение бесконечность (np.inf)
import math  #
from scipy.optimize import minimize  # берем готовую библиотечную функцию minimize
from scipy.optimize import LinearConstraint  # будем использовать линейные ограничения

In [ ]:
# описываем функцию, которую хотим минимизировать
def func_to_minimize(x):
    return (x - 2) * (x - 2) + 1 + 1 / (1 - x)

# Зададим начальное значение x
x = 0

# а теперь минимизируем нашу функцию
result = minimize(func_to_minimize, x)
print(result)  # выведем наш результа на экран

  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 6.0
        x: [ 0.000e+00]
      nit: 0
      jac: [-3.000e+00]
 hess_inv: [[1]]
     nfev: 174
     njev: 83


... то окажется, что минимум оказался в точке $0$. Не совсем тот результат, что мы ожидали.

Исправить положение можно либо изменив барьерную функцию так, чтобы она возрастала очень резко по мере приближения к границе запретной зоны, либо добавив специальный коэффициент $\alpha$, снизив влияние барьерной функции:

$$
f(x) = (x - 2)^2 + 1 + \alpha \cdot \frac {1} {1 - x}
$$

Попробуем:

In [ ]:
alpha = 0.1

# описываем функцию, которую хотим минимизировать
def func_to_minimize(x):
    return (x - 2) * (x - 2) + 1 + alpha * 1 / (1 - x)

# Зададим начальное значение x
x = 0

# а теперь минимизируем нашу функцию
result = minimize(func_to_minimize, x)
print(result)  # выведем наш результа на экран

  message: Desired error not necessarily achieved due to precision loss.
  success: False
   status: 2
      fun: 1.115695243467632
        x: [ 1.179e+00]
      nit: 2
      jac: [ 1.472e+00]
 hess_inv: [[ 1.595e-01]]
     nfev: 195
     njev: 95


Как видим, результат оказался гораздо лучше, хоть и неидеальным.

## Задание для самостоятельной работы

Изменить код в ячейке ниже (код основан на предыдущей работе) таким образом, чтобы учитывлось ограничение на время движения (необходимо затратить не более 0.028 часа), и расход топлива был минимальным, при условии использования штрафных или барьерных функций

In [12]:
import math
import numpy as np
from scipy.optimize import minimize

# исходные данные в километрах
h1 = 0.10  # 100 метров
h2 = 0.10  # 100 метров
l = 1      # 1000 метров

v1 = 40    # 40 км/ч
v2 = 30    # 30 км/ч

c1 = 0.115 # л/км (незасеянное поле)
c2 = 0.15  # л/км (засеянное поле)

# Параметр штрафной функции
alpha = 10000000  # большой коэффициент штрафа

# Функция расхода топлива
def fuel_cost(x):
    fuel1 = math.sqrt(h1**2 + x**2) * c2
    fuel2 = math.sqrt(h2**2 + (l - x)**2) * c1
    return fuel1 + fuel2

# Функция времени
def time_cost(x):
    t1 = math.sqrt(h1**2 + x**2) / v1
    t2 = math.sqrt(h2**2 + (l - x)**2) / v2
    return t1 + t2

# Целевая функция со штрафом
def func_to_minimize(x):
    # Основная цель - минимизировать расход топлива
    fuel = fuel_cost(x)

    # Вычисляем время
    time = time_cost(x)

    # Штраф за превышение ограничения по времени
    # Если время <= 0.028, штраф = 0
    # Если время > 0.028, штраф пропорционален квадрату превышения
    penalty = alpha * max(0, time - 0.028)**2

    return fuel + penalty

# Начальное приближение
x0 = 0.5

# Минимизация
result = minimize(func_to_minimize, x0, method='BFGS')

print("Результат оптимизации методом штрафных функций:")
print(result)
print(f"\nОптимальная точка пересечения x = {result.x[0]:.3f} км")
print(f"Расход топлива = {fuel_cost(result.x):.3f} л")
print(f"Время в пути = {time_cost(result.x):.5f} часа")
print(f"Ограничение по времени: {time_cost(result.x)} ≤ 0.028 -> {time_cost(result.x) <= 0.028}")



def penalty_method(alpha_start=1, alpha_multiplier=10, num_iterations=5):
    x = 0.5  # начальное приближение

    for i in range(num_iterations):
        alpha = alpha_start * (alpha_multiplier ** i)

        def func_with_penalty(x):
            fuel = fuel_cost(x)
            time = time_cost(x)
            penalty = alpha * max(0, time - 0.028)**2
            return fuel + penalty

        result = minimize(func_with_penalty, x, method='BFGS')
        x = result.x

        print(f"Итерация {i+1}, alpha = {alpha:.0f}")
        print(f"  x = {x[0]:.4f}, время = {time_cost(x):.4f}, штраф = {alpha * max(0, time_cost(x) - 0.028)**2:.6f}")

    return x

print("\n--- Итеративный метод штрафных функций ---")
optimal_x = penalty_method()
print(f"\nИтоговый результат: x = {optimal_x[0]:.3f} км")
print(f"Расход топлива = {fuel_cost(optimal_x):.3f} л")
print(f"Время в пути = {time_cost(optimal_x)} часа")




Результат оптимизации методом штрафных функций:
  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 0.14374578013941555
        x: [ 7.327e-01]
      nit: 6
      jac: [-5.588e-09]
 hess_inv: [[ 1.202e-03]]
     nfev: 16
     njev: 8

Оптимальная точка пересечения x = 0.733 км
Расход топлива = 0.144 л
Время в пути = 0.02800 часа
Ограничение по времени: 0.028000317234736304 ≤ 0.028 -> False

--- Итеративный метод штрафных функций ---
Итерация 1, alpha = 1
  x = 0.1180, время = 0.0335, штраф = 0.000030
Итерация 2, alpha = 10
  x = 0.1213, время = 0.0334, штраф = 0.000293
Итерация 3, alpha = 100
  x = 0.1556, время = 0.0330, штраф = 0.002467
Итерация 4, alpha = 1000
  x = 0.4664, время = 0.0300, штраф = 0.004087
Итерация 5, alpha = 10000
  x = 0.6911, время = 0.0283, штраф = 0.000784

Итоговый результат: x = 0.691 км
Расход топлива = 0.142 л
Время в пути = 0.02828002281536113 часа


/tmp/ipython-input-215/313463389.py:21: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fuel1 = math.sqrt(h1**2 + x**2) * c2
/tmp/ipython-input-215/313463389.py:22: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fuel2 = math.sqrt(h2**2 + (l - x)**2) * c1
/tmp/ipython-input-215/313463389.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  t1 = math.sqrt(h1**2 + x**2) / v1
/tmp/ipython-input-215/313463389.py:28: DeprecationWarning: Conversion of an array with ndim > 0 